<a href="https://colab.research.google.com/github/Abhinav9818/Github_repo_AI_assistant-RAG-/blob/main/Github_repo_AI_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
!pip -q install -U \
langchain \
langchain-community \
langchain-huggingface \
langchain-groq \
langchain-text-splitters \
faiss-cpu \
sentence-transformers \
gitpython

In [44]:
import os
from google.colab import userdata

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [45]:
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

print("Embedding model loaded successfully!")
print(llm.invoke("hi").content)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!
It's nice to meet you. Is there something I can help you with or would you like to chat?


In [46]:
!git clone https://github.com/langchain-ai/langchain.git
import os

print(os.path.exists("langchain"))

fatal: destination path 'langchain' already exists and is not an empty directory.
True


In [47]:
from pathlib import Path
SUPPORTED_EXTENSIONS = {
    ".py",
    ".js",
    ".ts",
    ".jsx",
    ".tsx",
    ".java",
    ".cpp",
    ".c",
    ".h",
    ".hpp",
    ".go",
    ".rs",
    ".md",
}

IGNORE_DIRS = {
    ".git",
    "__pycache__",
    "node_modules",
    "venv",
    ".venv",
    "dist",
    "build",
}

LANGUAGE_MAP = {
    ".py": "python",
    ".js": "javascript",
    ".ts": "typescript",
    ".jsx": "javascript",
    ".tsx": "typescript",
    ".java": "java",
    ".cpp": "cpp",
    ".c": "c",
    ".h": "c",
    ".hpp": "cpp",
    ".go": "go",
    ".rs": "rust",
    ".md": "markdown",
}
def load_repository(repo_path):
  document = []
  for root,dirs,files in os.walk(repo_path):
    dirs[:] = [d for d in dirs if d not in IGNORE_DIRS]
    for file in files:
      path = Path(root) / file
      if path.suffix not in SUPPORTED_EXTENSIONS:
        continue
      try:
        with open(path,"r",encoding = "utf-8") as f:
          text = f.read()
      except Exception:
        continue
      doc = Document(
          page_content = text,
          metadata = {
              "path" : str(path),
              "filename" : path.name,
              "language" : LANGUAGE_MAP.get(path.suffix,"unknown")
          }

      )
      document.append(doc)
  return document




documents = load_repository("langchain")
print(documents[0])
print(len(documents))
print(type(documents))


# currently each document holds one complete file
# we cannot embedd the whole file as one vector
# so now we devide each document into chunks -> embedd the chunks store them in vectors -> store the vector in FAISS

page_content='# Global development guidelines for the LangChain monorepo

This document provides context to understand the LangChain Python project and assist with development.

## Project architecture and context

### Monorepo structure

This is a Python monorepo with multiple independently versioned packages that use `uv`.

```txt
langchain/
├── libs/
│   ├── core/             # `langchain-core` primitives and base abstractions
│   ├── langchain/        # `langchain-classic` (legacy, no new features)
│   ├── langchain_v1/     # Actively maintained `langchain` package
│   ├── partners/         # Third-party integrations
│   │   ├── openai/       # OpenAI models and embeddings
│   │   ├── anthropic/    # Anthropic (Claude) integration
│   │   ├── ollama/       # Local model support
│   │   └── ... (other integrations maintained by the LangChain team)
│   ├── text-splitters/   # Document chunking utilities
│   ├── standard-tests/   # Shared test suite for integrations
│   ├── model-prof

In [48]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

chunks = text_splitter.split_documents(documents)
print(len(documents))
print(len(chunks))

2564
18933


In [49]:
# embeddings
from langchain_community.vectorstores import FAISS
vector_db = FAISS.from_documents(
    documents = chunks,
    embedding = embeddings
)
vector_db.save_local("repo_index")



# retriever

retriever = vector_db.as_retriever(
    search_kwargs = {"k" : 5}
)

results = retriever.invoke("where is chatGroq implemented")
print(results[0])

page_content='"""Groq integration for LangChain."""

from langchain_groq._version import __version__
from langchain_groq.chat_models import ChatGroq

__all__ = [
    "ChatGroq",
    "__version__",
]' metadata={'path': 'langchain/libs/partners/groq/langchain_groq/__init__.py', 'filename': '__init__.py', 'language': 'python'}


In [50]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template(
    """
    You are a senior software engineer.

   Answer the user's question ONLY using the provided code context.

If the answer cannot be found in the context, say:
"I couldn't find that information in the indexed repository."

Always mention the file paths when possible.

Context:
{context}

Question:
{question}
    """
)

In [51]:
def format_docs(docs):
  result = []
  for doc in docs:
    text = f"File : {doc.metadata['path']}\n\n{doc.page_content}"
    result.append(text)
  return "\n\n".join(result)



context = format_docs(results)
print(context)


File : langchain/libs/partners/groq/langchain_groq/__init__.py

"""Groq integration for LangChain."""

from langchain_groq._version import __version__
from langchain_groq.chat_models import ChatGroq

__all__ = [
    "ChatGroq",
    "__version__",
]

File : langchain/libs/core/langchain_core/tools/convert.py

Used to pass configuration that doesn't fit into standard tool fields.
            Chat models should process known extras when constructing model payloads.

            !!! example

                For example, Anthropic-specific fields like `cache_control`,
                `defer_loading`, or `input_examples`.

File : langchain/libs/langchain/langchain_classic/chat_loaders/__init__.py

"""**Chat Loaders** load chat messages from common communications platforms.

Load chat messages from various
communications platforms such as Facebook Messenger, Telegram, and
WhatsApp. The loaded chat messages can be used for fine-tuning models.
"""

File : langchain/libs/partners/groq/langchain_

In [52]:
response = llm.invoke(
    prompt.format(
        context = context,
        question = "where is chatGroq implemented"
    )
)
print(response.content)

The `ChatGroq` class is implemented in the file `langchain/libs/partners/groq/langchain_groq/chat_models.py`. It is also imported in `langchain/libs/partners/groq/langchain_groq/__init__.py` for easier access.


In [53]:
import ast
from langchain_core.documents import Document

def extract_python_chunks_pure_ast(doc):
  code = doc.page_content
  try :
    tree = ast.parse(code)
  except SyntaxError:
    return []
  lines = code.splitlines()
  chunks = []

  for node in tree.body:
    if isinstance(node,ast.ClassDef):
      start = node.lineno - 1
      end = node.end_lineno
      class_code = "\n".join(lines[start:end])
      chunks.append(
          Document(
              page_content = class_code,
              metadata = {
                  **doc.metadata,
                  "type" : "class",
                  "name" : node.name,
                  "start_line" : node.lineno,
                  "end_line" : node.end_lineno

              }
          )
      )
    elif isinstance(node,ast.FunctionDef):
      start = node.lineno - 1
      end = node.end_lineno
      func_code = "\n".join(lines[start:end])
      chunks.append(
          Document(
              page_content = func_code,
              metadata = {
                  **doc.metadata,
                  "type" : "function",
                  "name" : node.name,
                  "start" : node.lineno,
                  "end" : node.end_lineno
              }
          )
      )
  return chunks



In [54]:
""""
we have done a basic recursivecharactersplitter , a ast splitter that splits on the basis of
classes and functions now we are going for a hybrid approach in splitter where

we will split on the basis of classes and function and if the funct / class is bigger than 1000 len chars
then we will use recursivecharsplitter


"""

code_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)
def add_hybrid_chunk(chunks,page_content,metadata):
  doc = Document(
      page_content = page_content,
      metadata = metadata
  )
  if len(page_content) <= 1000:
    chunks.append(doc)
    return

  # for larger docs use charsplitter
  split_docs = code_splitter.split_documents([doc])
  for i,split_doc in enumerate(split_docs):
    split_doc.metadata["chunk"] = i+1
    split_doc.metadata["total_chunks"] = len(split_docs)
  chunks.extend(split_docs)




In [55]:
# the same ast code but we will add the add_hybrid_chunk function after both
# class insatnce and function instance
# and instead of appending in the chunk i will append using the add_hybrid_chunk function we created above


def extract_python_chunks_hybrid(doc):
  code = doc.page_content
  try :
    tree = ast.parse(code)
  except SyntaxError:
    return []
  lines = code.splitlines()
  chunks = []

  for node in tree.body:
    if isinstance(node,ast.ClassDef):
      start = node.lineno - 1
      end = node.end_lineno
      class_code = "\n".join(lines[start:end])
      metadata = {
          **doc.metadata,
          "type" : "class",
          "name" : node.name,
          "start_line" : node.lineno,
          "end_line" : node.end_lineno
      }

      add_hybrid_chunk(chunks,class_code,metadata)

    elif isinstance(node,ast.FunctionDef):
      start = node.lineno - 1
      end = node.end_lineno
      func_code = "\n".join(lines[start:end])
      metadata = {
                  **doc.metadata,
                  "type" : "function",
                  "name" : node.name,
                  "start" : node.lineno,
                  "end" : node.end_lineno
              }
      add_hybrid_chunk(chunks,func_code,metadata)
  return chunks



In [56]:
# recreating the chunks
python_chunks_pure_text_splitter = chunks
python_chunks_pure_ast = []
python_chunks_hybrid = []

for doc in documents:
  if doc.metadata["language"] == "python":
    python_chunks_pure_ast.extend(extract_python_chunks_pure_ast(doc))
    python_chunks_hybrid.extend(extract_python_chunks_hybrid(doc))
print(len(python_chunks_pure_ast))
print(len(python_chunks_hybrid))
print(len(chunks))



7511
16856
18933


In [57]:
# creating the faiss index again
# embedd vecry chunk in the vector form , store the vector
# and store the corresponding document to it
vector_db = FAISS.from_documents(
    documents = python_chunks_hybrid,
    embedding = embeddings

)

# now the retriever again
retriever = vector_db.as_retriever(
    search_kwargs = {"k" : 5}
)

# checking the retriever

results = retriever.invoke("where is groqchat implemented")
for doc in results:
  print(doc.page_content[:500])
  print()
  print(doc.metadata)
  print()

groq_api_base: str | None = Field(
        alias="base_url", default_factory=from_env("GROQ_API_BASE", default=None)
    )
    """Base URL path for API requests. Leave blank if not using a proxy or service
    emulator.
    """

    # to support explicit proxy for Groq
    groq_proxy: str | None = Field(default_factory=from_env("GROQ_PROXY", default=None))

    request_timeout: float | tuple[float, float] | Any | None = Field(
        default=None, alias="timeout"
    )
    """Timeout for reques

{'path': 'langchain/libs/partners/groq/langchain_groq/chat_models.py', 'filename': 'chat_models.py', 'language': 'python', 'type': 'class', 'name': 'ChatGroq', 'start_line': 126, 'end_line': 1332, 'chunk': 15, 'total_chunks': 63}

class ChatGroq(BaseChatModel):
    r"""Groq Chat large language models API.

    To use, you should have the
    environment variable `GROQ_API_KEY` set with your API key.

    Any parameters that are valid to be passed to the groq.create call
    can be passed in, e

In [58]:
# we have already created the prompt earlier

# now we check what the llm responds finally again


question = "where is chatgroq implemented and how do i increase my hieght  give suggestion for the hieght thing on your own dont search the repo for that "

context = retriever.invoke(question)

llm_response = llm.invoke(
    prompt.format(
        context = context,
        question = question
    )
)
print(llm_response)
print(llm_response.content)


# it can ans both the questions
# 1 -- the ones related with the github repo answred through the context
# 2 -- the ones that are completely unrelated to the context

#  any github repo link consisting of programms expecially of python will work

# future changes -- make it friendly for all the programs not only python



content="The ChatGroq class is implemented in the file located at 'langchain/libs/partners/groq/langchain_groq/chat_models.py'.\n\nAs for increasing your height, I'd like to suggest some general tips that are not related to the code repository: \n1. Maintain good posture to make the most of your current height.\n2. Engage in exercises that promote spinal flexibility and strength, such as yoga or Pilates.\n3. Ensure you're getting enough sleep, as growth hormone is released during deep sleep.\n4. Eat a balanced diet rich in nutrients, particularly those that support bone health like calcium and vitamin D.\n5. Consider incorporating activities that promote bone density, such as weight-bearing exercises or sports. \n\nPlease consult a healthcare professional for personalized advice on improving your height." additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 161, 'prompt_tokens': 1536, 'total_tokens': 1697, 'completion_time': 0.501095027, 'completion_tokens_detai